Load product-substitute data.

In [ ]:
import pandas as pd

# Convert CSV files into dataframes
substitutes_df = pd.read_csv("../data/dept-sample/obj1/beverages_substitutes.csv")
product_info_df = pd.read_csv('../data/intermediary/product-info.csv')
orders_df = pd.read_csv('../data/intermediary/order-products-enhanced.csv')

In [ ]:

import numpy as np
import statsmodels.api as sm
from scipy.sparse import csr_matrix
from tqdm import tqdm

def compute_transferability(orders_df, substitutes_df, product_info, output_csv=None, top_n=None):
    
    if output_csv:
        with open(output_csv, 'w') as f:
            f.write("product_id,substitute_id,score,rank,raw_dtr,ame,adjusted_dtr,same_aisle,same_department\n")
    
    # Precompute order × product presence matrix
    orders_df['order_idx'] = orders_df['order_id'].astype('category').cat.codes
    orders_df['product_idx'] = orders_df['product_id'].astype('category').cat.codes
    order_idx_map = dict(enumerate(orders_df['order_id'].astype('category').cat.categories))
    product_idx_map = dict(enumerate(orders_df['product_id'].astype('category').cat.categories))
    
    n_orders = orders_df['order_idx'].max() + 1
    n_products = orders_df['product_idx'].max() + 1
    
    # Sparse matrix: rows=orders, cols=products
    data = np.ones(len(orders_df), dtype=np.uint8)
    order_product_matrix = csr_matrix((data, (orders_df['order_idx'], orders_df['product_idx'])),
                                     shape=(n_orders, n_products))
    
    # Map product_id → aisle/department
    aisle_map = product_info.set_index('product_id')['aisle'].to_dict()
    dept_map = product_info.set_index('product_id')['department'].to_dict()
    
    # Keep unique order-level features
    order_features = orders_df[['order_idx', 'order_size_cat', 'orders_per_user_cat']].drop_duplicates('order_idx').set_index('order_idx')
    
    for product_id, group in tqdm(substitutes_df.groupby('product_id'), desc="Processing products"):
        if top_n:
            group = group.nsmallest(top_n, 'rank')
        
        if product_id not in product_idx_map.values():
            continue
        idx_A = {k:v for v,k in product_idx_map.items()}[product_id]
        orders_A_idx = set(order_product_matrix[:, idx_A].nonzero()[0])
        if not orders_A_idx:
            continue
        
        for _, row in group.iterrows():
            B = row['substitute_id']
            if B not in product_idx_map.values():
                continue
            idx_B = {k:v for v,k in product_idx_map.items()}[B]
            orders_B_idx = set(order_product_matrix[:, idx_B].nonzero()[0])
            
            raw_sales_B_without_A = len(orders_B_idx - orders_A_idx)
            raw_dtr = raw_sales_B_without_A / len(orders_A_idx) if orders_A_idx else 0
            
            # Construct per-order feature matrix for logistic regression
            X = order_features.copy()
            X['A_absent'] = (~X.index.isin(orders_A_idx)).astype(int)
            X['same_aisle'] = int(aisle_map.get(product_id) == aisle_map.get(B))
            X['same_department'] = int(dept_map.get(product_id) == dept_map.get(B))
            X = sm.add_constant(X)
            
            y = X.index.isin(orders_B_idx).astype(int)
            
            try:
                logit_model = sm.Logit(y, X).fit(disp=0)
            except:
                continue
            
            # Compute AME
            X_present = X.copy(); X_present['A_absent'] = 0
            X_absent = X.copy(); X_absent['A_absent'] = 1
            pred_present = logit_model.predict(X_present)
            pred_absent = logit_model.predict(X_absent)
            ame = (pred_absent - pred_present).mean()
            
            adjusted_dtr = raw_dtr * ame
            
            result = {
                'product_id': product_id,
                'substitute': B,
                'score': row['score'],
                'rank': row['rank'],
                'raw_dtr': raw_dtr,
                'ame': ame,
                'adjusted_dtr': adjusted_dtr,
                'same_aisle': int(X['same_aisle'].iloc[0]),
                'same_department': int(X['same_department'].iloc[0]),
                'coef': logit_model.params['target_present'],
                'p_value': logit_model.pvalues['target_present'],
                'pseudo_r2': 1 - (logit_model.llf / logit_model.llnull),
            }
            
            if output_csv:
                pd.DataFrame([result]).to_csv(output_csv, mode='a', index=False, header=False)
            else:
                return result  
    

compute_transferability(orders_df, 
    substitutes_df, 
    product_info_df, 
    output_csv="../data/dept-sample/obj2/beverages_transfer.csv",
    top_n=10
    )


Validation

In [ ]:
from scipy.stats import spearmanr

def validate_adjusted_rank_stability(subs_objective1, transfer_df):
    results = []

    for product_A in subs_objective1['product'].unique():
        df1 = subs_objective1[subs_objective1['product'] == product_A].set_index('substitute')
        df2 = transfer_df[transfer_df['product'] == product_A].set_index('substitute')

        # Align substitutes
        common_subs = df1.index.intersection(df2.index)
        if len(common_subs) < 2:
            results.append({'product': product_A, 'rho': np.nan, 'pval': np.nan})
            continue

        rank1 = df1.loc[common_subs, 'rank']
        rank2 = df2.loc[common_subs, 'adjusted_dtr'].rank(ascending=False)  # rank by adjusted_dtr

        rho, pval = spearmanr(rank1, rank2)
        results.append({'product': product_A, 'rho': rho, 'pval': pval})

    return pd.DataFrame(results)

substitutes_df = pd.read_csv("../data/dept-sample/obj1/beverages_substitutes.csv")
transfer_df = pd.read_csv("../data/dept-sample/obj2/beverages_transfer.csv")

rank_stability_df = validate_adjusted_rank_stability(substitutes_df, transfer_df)
print(rank_stability_df.head())

rank_stability_df.to_csv("../data/dept-sample/obj2/beverages_transfer_result.csv", index=False)